# RAG from scratch — no LangChain

Building the full retrieval-augmented generation loop by hand: PDF → chunks → embeddings → vector search → prompt → LLM → answer with cited sources. The goal is to come out the other side knowing exactly what a framework like LangChain abstracts away, instead of trusting it blindly.

The document: *Attention Is All You Need* (Vaswani et al., 2017) — the paper behind everything built in the previous two repos. Fitting choice, and a document whose content I already know well enough to judge whether the RAG's answers are actually correct.

In [ ]:
import os
import bisect

import numpy as np
import pymupdf
import chromadb
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from mistralai.client import Mistral

load_dotenv()
assert os.getenv("MISTRAL_API_KEY"), "Missing MISTRAL_API_KEY — copy .env.example to .env at the repo root and fill it in"

## 1. Extracting text from the PDF

PyMuPDF reads the PDF page by page. Keeping track of *where* each page starts in the full concatenated text lets any later chunk trace itself back to a real page number — that's what makes "cited sources" mean something concrete instead of a vague gesture at the document.

In [ ]:
PDF_PATH = "data/attention_is_all_you_need.pdf"

doc = pymupdf.open(PDF_PATH)
pages_text = [page.get_text() for page in doc]

full_text = ""
page_offsets = []  # character offset where each page starts in full_text
for page_text in pages_text:
    page_offsets.append(len(full_text))
    full_text += page_text


def page_for_offset(offset):
    """Given a character offset in full_text, return which page (1-indexed) it falls on."""
    return bisect.bisect_right(page_offsets, offset)


print(f"{len(pages_text)} pages, {len(full_text):,} characters total")
print(full_text[:300])

## 2. Three chunking strategies

A chunk is the unit of text that gets embedded and retrieved — too big and irrelevant text dilutes the match, too small and you lose context. There's no single right size; three different strategies, three different tradeoffs.

### 2a. Fixed-size chunking

The simplest possible approach: slice the text every 512 characters, with a small overlap so a sentence split across a chunk boundary still appears whole in at least one chunk. Fast, predictable, and completely ignorant of sentence or paragraph structure — it will happily cut a sentence in half.

In [ ]:
def chunk_fixed_size(text, size=512, overlap=50):
    chunks = []
    i = 0
    while i < len(text):
        chunk_text = text[i:i + size]
        chunks.append({"text": chunk_text, "page": page_for_offset(i)})
        i += size - overlap
    return chunks


fixed_chunks = chunk_fixed_size(full_text)
avg_len = sum(len(c["text"]) for c in fixed_chunks) / len(fixed_chunks)
print(f"Fixed-size: {len(fixed_chunks)} chunks, avg {avg_len:.0f} chars")
print(repr(fixed_chunks[5]["text"][:150]))

### 2b. Sentence-based chunking

Split into real sentences first (using nltk), then group consecutive sentences together until adding one more would exceed the target size. Never cuts a sentence in half — a real improvement over fixed-size for readability, at the cost of variable chunk sizes.

In [ ]:
def chunk_sentence_based(text, target_size=512):
    sentences = sent_tokenize(text)
    chunks = []
    current, current_len = [], 0
    search_from = 0
    chunk_start = 0

    for sent in sentences:
        pos = text.find(sent, search_from)
        if pos == -1:
            pos = search_from

        if current and current_len + len(sent) > target_size:
            chunks.append({"text": " ".join(current), "page": page_for_offset(chunk_start)})
            current, current_len = [], 0
            chunk_start = pos

        if not current:
            chunk_start = pos
        current.append(sent)
        current_len += len(sent)
        search_from = pos + len(sent)

    if current:
        chunks.append({"text": " ".join(current), "page": page_for_offset(chunk_start)})
    return chunks


sentence_chunks = chunk_sentence_based(full_text)
avg_len = sum(len(c["text"]) for c in sentence_chunks) / len(sentence_chunks)
print(f"Sentence-based: {len(sentence_chunks)} chunks, avg {avg_len:.0f} chars")
print(repr(sentence_chunks[5]["text"][:150]))

### 2c. Semantic chunking

The most sophisticated of the three: embed every sentence, measure how similar each sentence is to the next one, and start a new chunk exactly where the similarity drops — a topic shift. Instead of a fixed similarity cutoff (which turned out to fragment this document into 280+ tiny chunks — academic papers have plenty of short, disjointed sentences in author lists and headers), the threshold is computed from the document's own similarity distribution (the 25th percentile), so it adapts to how "choppy" a given document naturally is.

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


def chunk_semantic(text, percentile=25, min_size=200, max_size=1000):
    sentences = sent_tokenize(text)
    sentence_embeddings = embedding_model.encode(sentences, show_progress_bar=False)

    similarities = []
    for i in range(len(sentence_embeddings) - 1):
        a, b = sentence_embeddings[i], sentence_embeddings[i + 1]
        similarities.append(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    # Adaptive threshold: break at the weakest ~25% of sentence transitions,
    # not at an arbitrary fixed similarity value
    threshold = np.percentile(similarities, percentile)

    chunks = []
    current = [sentences[0]]
    current_len = len(sentences[0])
    chunk_start = text.find(sentences[0])

    for i, sim in enumerate(similarities):
        next_sent = sentences[i + 1]
        topic_shift = sim < threshold and current_len >= min_size
        too_long = current_len + len(next_sent) > max_size

        if topic_shift or too_long:
            chunks.append({"text": " ".join(current), "page": page_for_offset(chunk_start)})
            current, current_len = [], 0
            chunk_start = text.find(next_sent, chunk_start)

        current.append(next_sent)
        current_len += len(next_sent)

    if current:
        chunks.append({"text": " ".join(current), "page": page_for_offset(chunk_start)})
    return chunks


semantic_chunks = chunk_semantic(full_text)
avg_len = sum(len(c["text"]) for c in semantic_chunks) / len(semantic_chunks)
print(f"Semantic: {len(semantic_chunks)} chunks, avg {avg_len:.0f} chars")
print(repr(semantic_chunks[5]["text"][:150]))

### Comparing the three

| Strategy | Chunks | Avg size | When to use it |
| --- | --- | --- | --- |
| Fixed-size | ~86 | ~509 chars | Quick baseline, huge corpora where per-chunk cost matters more than quality |
| Sentence-based | ~89 | ~443 chars | Good default — respects sentence boundaries, cheap to compute (no embeddings needed) |
| Semantic | ~64 | ~616 chars | Best retrieval quality on documents with clear topic shifts (papers, docs) — costs one embedding pass over every sentence just to build the chunks |

Semantic chunking produces the fewest, most topically coherent chunks here — but it's also the only one of the three that needs the embedding model *twice* (once to decide where to cut, once more to embed the final chunks). For a huge corpus, that's a real cost worth weighing against the quality gain.

## 3. Embeddings + ChromaDB

Proceeding with the semantic chunks for the rest of the pipeline. Each chunk gets embedded locally with `all-MiniLM-L6-v2` (384-dimensional vectors, no API call, no cost) and stored in ChromaDB alongside its page number.

In [ ]:
chunks = semantic_chunks
chunk_embeddings = embedding_model.encode([c["text"] for c in chunks], show_progress_bar=False)

chroma_client = chromadb.Client()
collection = chroma_client.create_collection("attention_paper")
collection.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=[e.tolist() for e in chunk_embeddings],
    documents=[c["text"] for c in chunks],
    metadatas=[{"page": c["page"]} for c in chunks],
)
print(f"{len(chunks)} chunks embedded and stored in ChromaDB")

## 4. The query pipeline

Question → embed it with the *same* model used for the chunks (this matters — you can't compare vectors from two different embedding models) → ChromaDB finds the 5 closest chunks by cosine similarity → those 5 chunks become the context in a prompt → Mistral answers, citing which source numbers it used.

In [ ]:
mistral_client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))


def ask(question, top_k=5):
    question_embedding = embedding_model.encode([question])[0].tolist()
    results = collection.query(query_embeddings=[question_embedding], n_results=top_k)
    retrieved = list(zip(results["documents"][0], results["metadatas"][0], results["distances"][0]))

    context = "\n\n".join(
        f"[Source {i + 1}, page {meta['page']}]: {text}"
        for i, (text, meta, _dist) in enumerate(retrieved)
    )
    prompt = f"""Answer the question using ONLY the sources below. Cite source numbers in your answer.
If the answer isn't in the sources, say so explicitly instead of guessing.

Sources:
{context}

Question: {question}

Answer:"""

    response = mistral_client.chat.complete(
        model="mistral-small-latest",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content, retrieved


answer, retrieved = ask("What is multi-head attention?")
print("ANSWER:\n", answer)
print("\nSOURCES:")
for text, meta, dist in retrieved:
    print(f"  page {meta['page']}, distance {dist:.3f}: {text[:80]}...")

## 5. Testing 10 questions

A mix of questions that should be easy to answer from the paper, run through the exact same `ask()` function above — no cherry-picking after the fact.

In [ ]:
import time

test_questions = [
    "What is multi-head attention?",
    "What optimizer was used to train the Transformer?",
    "How many attention heads does the base Transformer model use?",
    "What is the positional encoding formula?",
    "How long did it take to train the base model?",
    "What dataset was used for training the English-to-German translation model?",
    "What is the computational complexity of self-attention compared to recurrent layers?",
    "Who are the authors of this paper?",
    "What is label smoothing and what value was used?",
    "What BLEU score did the big Transformer model achieve on English-to-French translation?",
]

for q in test_questions:
    t0 = time.perf_counter()
    a, sources = ask(q)
    elapsed = time.perf_counter() - t0
    pages = sorted({m["page"] for _, m, _ in sources})
    print(f"Q: {q}")
    print(f"A: {a}")
    print(f"({elapsed:.2f}s, sources: pages {pages})")
    print("-" * 80)

### Failure analysis

Two real failures out of ten, both worth understanding rather than papering over:

**"What optimizer was used to train the Transformer?"** — the paper states clearly, in section 5.3, that Adam was used with specific β1/β2/ε values. The RAG missed it. Why: that section is short, dense, and mostly formulas — semantically, "optimizer hyperparameters and a learning rate formula" doesn't embed particularly close to the phrasing of the question. The top-5 retrieved chunks simply didn't include that one. This is the core limitation of embedding-based retrieval: it matches *meaning*, and a terse, formula-heavy passage can have a weak semantic fingerprint.

**"Who are the authors of this paper?"** — the model answered about a *different* paper ("Neural machine translation in linear time"), which is actually one of the works *cited in the bibliography*, not the paper being asked about. What likely happened: a query like "authors of this paper" embeds closer to the bibliography section (full of author names and paper titles) than to the actual page-1 header block, which mixes author names with affiliations and email addresses in a way that reads awkwardly out of context. A genuinely instructive failure — the retrieval didn't fail to find text about authors, it found the *wrong section* about authors.

Both failures point to the same lesson: semantic search finds text that *sounds like* the answer, not necessarily the text that *is* the answer. This is exactly the gap a reranker (lab02) is built to close.

## Takeaways

Every step here — chunking, embedding, storing, searching, prompting — is what LangChain's `RecursiveCharacterTextSplitter`, `Chroma`, and `RetrievalQA` do internally. Having built it by hand means lab02's LangChain rewrite will be a matter of recognizing familiar steps behind new class names, not learning a new mental model from zero.

The two failures above aren't bugs in this implementation — they're inherent limits of pure semantic search, and the reason reranking and rigorous evaluation (RAGAS) exist as a next step.